# Markov Decision Processes

**Companion lesson:** https://ml-viz.vercel.app/courses/reinforcement-learning/01-markov-decision-processes

A from-scratch, runnable implementation of the concepts in the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## A gridworld MDP

We define the environment, then solve it with **value iteration** — repeatedly applying the Bellman optimality update until the value function stops changing.

In [ ]:
class GridWorld:
    """n x n grid. Start top-left (0), goal bottom-right. Actions: 0=up 1=down 2=left 3=right.
    Reward -1 per step, +10 at the goal (terminal)."""
    def __init__(self, n=5):
        self.n = n; self.nS = n*n; self.nA = 4; self.goal = n*n-1
    def step(self, s, a):
        r, c = divmod(s, self.n)
        if a==0: r = max(0, r-1)
        elif a==1: r = min(self.n-1, r+1)
        elif a==2: c = max(0, c-1)
        else: c = min(self.n-1, c+1)
        s2 = r*self.n + c
        done = (s2 == self.goal)
        return s2, (10.0 if done else -1.0), done

env = GridWorld(5)
print('states:', env.nS, '| actions:', env.nA, '| goal:', env.goal)

## Value iteration

$V(s)\leftarrow\max_a\big[r(s,a)+\gamma V(s')\big]$ for the (deterministic) gridworld, iterated to convergence.

In [ ]:
def value_iteration(env, gamma=0.9, tol=1e-6):
    V = np.zeros(env.nS)
    for it in range(1000):
        V_new = V.copy()
        for s in range(env.nS):
            if s == env.goal: continue
            V_new[s] = max(env.step(s,a)[1] + gamma*V[env.step(s,a)[0]] for a in range(env.nA))
        if np.max(np.abs(V_new - V)) < tol:
            print(f'converged in {it} iterations'); V = V_new; break
        V = V_new
    return V

V = value_iteration(env)
print('V* grid:'); print(np.round(V.reshape(5,5), 1))

## Extract and visualize the optimal policy

The greedy policy w.r.t. $V^*$ — the best action in every cell.

In [ ]:
def greedy_policy(env, V, gamma=0.9):
    pi = np.zeros(env.nS, int)
    for s in range(env.nS):
        pi[s] = np.argmax([env.step(s,a)[1] + gamma*V[env.step(s,a)[0]] for a in range(env.nA)])
    return pi

arrows = {0:'↑',1:'↓',2:'←',3:'→'}
pi = greedy_policy(env, V)
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(V.reshape(5,5), cmap='viridis')
for s in range(env.nS):
    r,c = divmod(s, env.n)
    ax.text(c, r, 'G' if s==env.goal else arrows[pi[s]], ha='center', va='center', color='w', fontsize=16)
ax.set_title('Optimal value (color) + policy (arrows)'); ax.axis('off'); plt.show()

## Discounting shapes the values

In [ ]:
for g in [0.5, 0.9, 0.99]:
    Vg = value_iteration(env, gamma=g)
    print(f'gamma={g}: V(start)={Vg[0]:.2f}')

## Key takeaways

- An MDP is states, actions, transitions, rewards, and a discount $\gamma$.
- **Value iteration** applies the Bellman optimality update until $V$ converges.
- The optimal policy is **greedy** with respect to $V^*$.
- $\gamma$ sets the horizon: larger $\gamma$ values distant rewards more.